# STUDY ASSISTANT v2.0 - Wk10
# PariShiksha | NCERT Gravitation (iesc110)

In [6]:
import os

folders = ['data', 'outputs']
files = [
    'wk10_chunks.json',
    'chunking_diff.md',
    'retrieval_log.json',
    'retrieval_misses.md',
    'prompt_diff.md',
    'eval_scored.csv',
    'eval_v2_scored.csv',
    'fix_memo.md',
    'reflection.md',
    'README.md'
]

In [5]:
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created folder: {folder}")

for file in files:
    if not os.path.exists(file):
        open(file, 'w').close()
        print(f"Created file: {file}")
    else:
        print(f"Already exists: {file}")

print("\n Project structure ready!")

Created folder: data
Created folder: outputs
Created file: wk10_chunks.json
Created file: chunking_diff.md
Created file: retrieval_log.json
Created file: retrieval_misses.md
Created file: prompt_diff.md
Created file: eval_scored.csv
Created file: eval_v2_scored.csv
Created file: fix_memo.md
Created file: reflection.md
Created file: README.md

 Project structure ready!


In [15]:
import sys
!{sys.executable} -m pip install PyMuPDF

  Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl.metadata (24 kB)
Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl (19.2 MB)


In [17]:
import fitz

doc = fitz.open("data/iesc110.pdf")

print(f"Total pages: {len(doc)}")
print(f"\n--- Checking all page headings ---\n")

for i, page in enumerate(doc):
    text = page.get_text()
    preview = text[:150].replace('\n', ' ')
    print(f"Page {i+1}: {preview}")
    print()

Total pages: 24

--- Checking all page headings ---

Page 1: Sound Waves:  Characteristics and  Applications Chapter  10 Sound is an everyday sensory experience that helps us become  aware of our surroundings. E

Page 2: Sound Waves: Characteristics and Applications 185 Activity 10.1: Let us explore 1.	 Take a cardboard box with one side open  and a rubber band. 2.	 St

Page 3: 186 Exploration|Grade 9 10.1.1 Tuning fork An instrument that is often used for experiments with sound is a tuning  fork. A tuning fork is a U-shaped 

Page 4: Sound Waves: Characteristics and Applications 187 2.	 Now, place your ear against the desk, close  your other ear and listen again, as shown  in Fig. 

Page 5: 188 Exploration|Grade 9 3.	 Assertion (A): We cannot hear the sound of a bell ringing in a closed jar  after most of the air is pumped out. 	 Reason (

Page 6: Sound Waves: Characteristics and Applications 189 Fig. 10.9: Density of air in a tube with a piston (a) Piston not oscillating (average air

# STAGE 1 — Chunking
# Goal: Split iesc110 PDF into metadata-rich chunks

In [18]:
import fitz          # PDF extraction
import json          # saving chunks
import re            # pattern matching for content types
import tiktoken      # token counting
from collections import Counter

print("All libraries imported successfully")

All libraries imported successfully


In [19]:
tokenizer = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(tokenizer.encode(text))

# Quick test
test = "Sound is a form of energy that travels through a medium."
print(f"Test sentence: '{test}'")
print(f"Token count: {count_tokens(test)}")
print("Tokenizer ready")

Test sentence: 'Sound is a form of energy that travels through a medium.'
Token count: 12
Tokenizer ready


In [25]:
def get_content_type(text):
    text_lower = text.lower().strip()
    
    
    if re.search(r'(^example\s+\d|example\s+\d+\.|worked example|solution\s*:|let us calculate)', text_lower):
        return "worked_example"
    
    
    elif re.search(r'(activity\s+\d+|exercise\s+\d+|\d+\.\s+which|assertion\s*\(a\)|try this|do you know)', text_lower):
        return "question_or_exercise"
    
    else:
        return "prose"

# Test on 3 sentences
tests = [
    ("Example 10.1: Calculate the wavelength.", "worked_example"),
    ("Activity 10.1: Take a cardboard box.", "question_or_exercise"),
    ("Sound travels faster in solids than gases.", "prose"),
    ("For example, grasshoppers rub their wings.", "prose"),  
    ("Sound Waves: Characteristics and Applications Chapter 10", "prose"), 
]

print("--- Classifier test results ---\n")
all_passed = True
for text, expected in tests:
    result = get_content_type(text)
    status = "Yes" if result == expected else "No"
    if result != expected:
        all_passed = False
    print(f"{status} Expected: {expected}")
    print(f"   Got:      {result}")
    print(f"   Text:     '{text[:60]}'")
    print()

if all_passed:
    print("All tests passed! Classifier improved.")
else:
    print("Some tests still failing — let's fix further.")

--- Classifier test results ---

Yes Expected: worked_example
   Got:      worked_example
   Text:     'Example 10.1: Calculate the wavelength.'

Yes Expected: question_or_exercise
   Got:      question_or_exercise
   Text:     'Activity 10.1: Take a cardboard box.'

Yes Expected: prose
   Got:      prose
   Text:     'Sound travels faster in solids than gases.'

Yes Expected: prose
   Got:      prose
   Text:     'For example, grasshoppers rub their wings.'

Yes Expected: prose
   Got:      prose
   Text:     'Sound Waves: Characteristics and Applications Chapter 10'

All tests passed! Classifier improved.


In [29]:
def chunk_pdf(pdf_path, max_tokens=250):
    doc = fitz.open(pdf_path)
    chunks = []
    chunk_id = 0

    for page_num, page in enumerate(doc):
        text = page.get_text()

        # Split page into paragraphs
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

        current_chunk = ""
        current_tokens = 0

        for para in paragraphs:
            para_tokens = count_tokens(para)

            # If paragraph itself is too big, split by sentences
            if para_tokens > max_tokens:
                sentences = re.split(r'(?<=[.!?])\s+', para)
                for sentence in sentences:
                    sent_tokens = count_tokens(sentence)
                    if current_tokens + sent_tokens > max_tokens:
                        if current_chunk.strip():
                            chunks.append({
                                "chunk_id": f"chunk_{chunk_id:04d}",
                                "source": pdf_path,
                                "page": page_num + 1,
                                "content_type": get_content_type(current_chunk),
                                "text": current_chunk.strip(),
                                "token_count": current_tokens
                            })
                            chunk_id += 1
                        current_chunk = sentence
                        current_tokens = sent_tokens
                    else:
                        current_chunk += " " + sentence
                        current_tokens += sent_tokens
            else:
                # Paragraph fits — add to current chunk or start new one
                if current_tokens + para_tokens > max_tokens:
                    if current_chunk.strip():
                        chunks.append({
                            "chunk_id": f"chunk_{chunk_id:04d}",
                            "source": pdf_path,
                            "page": page_num + 1,
                            "content_type": get_content_type(current_chunk),
                            "text": current_chunk.strip(),
                            "token_count": current_tokens
                        })
                        chunk_id += 1
                    current_chunk = para
                    current_tokens = para_tokens
                else:
                    current_chunk += "\n\n" + para
                    current_tokens += para_tokens

        # Save last chunk of the page
        if current_chunk.strip():
            chunks.append({
                "chunk_id": f"chunk_{chunk_id:04d}",
                "source": pdf_path,
                "page": page_num + 1,
                "content_type": get_content_type(current_chunk),
                "text": current_chunk.strip(),
                "token_count": current_tokens
            })
            chunk_id += 1

    return chunks

print("Chunking function defined and ready")

Chunking function defined and ready


In [30]:
chunks = chunk_pdf("data/iesc110.pdf")

print(f"Total chunks created: {len(chunks)}")
print(f"\n Content type breakdown")
types = Counter(c['content_type'] for c in chunks)
for t, count in types.items():
    print(f"  {t}: {count}")

print(f"\n Token size stats")
token_counts = [c['token_count'] for c in chunks]
print(f"  Min tokens: {min(token_counts)}")
print(f"  Max tokens: {max(token_counts)}")
print(f"  Avg tokens: {sum(token_counts)//len(token_counts)}")

Total chunks created: 69

 Content type breakdown
  prose: 50
  question_or_exercise: 13
  worked_example: 6

 Token size stats
  Min tokens: 24
  Max tokens: 250
  Avg tokens: 191


In [31]:
for content_type in ['prose', 'worked_example', 'question_or_exercise']:
    print(f"\n{'='*50}")
    print(f"CONTENT TYPE: {content_type}")
    print(f"{'='*50}")
    
    for c in chunks:
        if c['content_type'] == content_type:
            print(f"chunk_id : {c['chunk_id']}")
            print(f"page     : {c['page']}")
            print(f"tokens   : {c['token_count']}")
            print(f"text preview:\n{c['text'][:250]}")
            break


CONTENT TYPE: prose
chunk_id : chunk_0000
page     : 1
tokens   : 191
text preview:
Sound Waves: 
Characteristics and 
Applications
Chapter 
10
Sound is an everyday sensory experience that helps us become 
aware of our surroundings. Every day, we hear a variety of sounds 
in our surroundings, such as human voices, birds chirping, 
w

CONTENT TYPE: worked_example
chunk_id : chunk_0032
page     : 12
tokens   : 231
text preview:
Sound Waves: Characteristics and Applications
195
Density
Distance
Amplitude
Density
Distance
Amplitude
C 
R 
C 
R 
C 
R
C 
R 
C 
R 
C 
R
Fig. 10.20: Sound waves
(a) Low amplitude
(b) High amplitude
In this activity, you would have noticed that the f

CONTENT TYPE: question_or_exercise
chunk_id : chunk_0002
page     : 2
tokens   : 241
text preview:
Sound Waves: Characteristics and Applications
185
Activity 10.1: Let us explore
1. Take a cardboard box with one side open 
and a rubber band. 2. Stretch the rubber band across the open 
side of the box (Fig. 10.2). 3.

In [32]:
with open("wk10_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

# Verify it saved correctly
with open("wk10_chunks.json", "r", encoding="utf-8") as f:
    saved = json.load(f)

print(f"Saved {len(saved)} chunks to wk10_chunks.json")
print(f"\n--- First chunk preview ---")
print(json.dumps(saved[0], indent=2)[:400])

Saved 69 chunks to wk10_chunks.json

--- First chunk preview ---
{
  "chunk_id": "chunk_0000",
  "source": "data/iesc110.pdf",
  "page": 1,
  "content_type": "prose",
  "text": "Sound Waves: \nCharacteristics and \nApplications\nChapter \n10\nSound is an everyday sensory experience that helps us become \naware of our surroundings. Every day, we hear a variety of sounds \nin our surroundings, such as human voices, birds chirping, \nwaves crashing on the seashore
